# E-commerce Sales Data Processing - pipeline

Runs the full pipeline: **unit tests -> raw tables -> enriched customers/products -> enriched orders -> profit aggregate -> reconciliation -> SQL reports**.

* Transformation logic: `scripts/transformations.py` (PySpark only)
* Orchestration and reconciliation: `scripts/pipeline.py`
* Tests: `tests/test_transformations.py`
* Design, data dictionary and data quality rules: `docs/`

Set the widgets at the top, then **Run all**.

In [0]:
import os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

dbutils.widgets.text("catalog", "dev_test", "Catalog")
dbutils.widgets.text("schema", "ecommerce_sales", "Schema")
# Relative paths are resolved from this notebook's folder; absolute paths (e.g. /Volumes/...) are used as-is
dbutils.widgets.text("data_path", "../data", "Source data folder")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
DATA_PATH_INPUT = dbutils.widgets.get("data_path")
DATA_PATH = (DATA_PATH_INPUT if os.path.isabs(DATA_PATH_INPUT)
             else os.path.abspath(os.path.join(os.getcwd(), DATA_PATH_INPUT)))

# Fail fast with a clear message if the source files are not where we expect them
REQUIRED_FILES = ["Orders.json", "Customer.xlsx", "Products.csv"]
missing = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(DATA_PATH, f))]
if missing and DATA_PATH.startswith("/Workspace/"):
    raise FileNotFoundError(
        f"Missing {missing} in data_path '{DATA_PATH_INPUT}'. Copy the source files into the repo "
        f"data/ folder or set the 'data_path' widget to the folder that contains them.")

# Print the widget value, not the absolute path (which contains the workspace user name)
print(f"catalog={CATALOG}  schema={SCHEMA}  data_path={DATA_PATH_INPUT}")

In [0]:
%run ./02_run_unit_tests

In [0]:
import importlib
import logging
import sys

sys.dont_write_bytecode = True
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
logging.basicConfig(level=logging.INFO)

from scripts import transformations, pipeline
importlib.reload(transformations)   # pick up code edits without restarting Python
importlib.reload(pipeline)
from scripts.pipeline import PipelineConfig, run_pipeline

cfg = PipelineConfig(catalog=CATALOG, schema=SCHEMA, data_path=DATA_PATH)

In [0]:
summary = run_pipeline(spark, cfg)
for k, v in summary.items():
    print(f"{k:30s} {v}")

In [0]:
for t in ["enriched_customers", "enriched_products", "enriched_orders", "agg_profit"]:
    print(t)
    display(spark.table(cfg.table(t)).limit(10))

## Task 5: SQL aggregates
The queries read `agg_profit` from the catalog and schema set in the widgets.

In [0]:
%sql
SELECT order_year, SUM(total_profit) AS profit
FROM IDENTIFIER(:catalog || '.' || :schema || '.agg_profit')
GROUP BY order_year
ORDER BY order_year

In [0]:
%sql
SELECT order_year, category, SUM(total_profit) AS profit
FROM IDENTIFIER(:catalog || '.' || :schema || '.agg_profit')
GROUP BY order_year, category
ORDER BY order_year, category

In [0]:
%sql
SELECT customer_id, customer_name, SUM(total_profit) AS profit
FROM IDENTIFIER(:catalog || '.' || :schema || '.agg_profit')
GROUP BY customer_id, customer_name
ORDER BY profit DESC

In [0]:
%sql
SELECT customer_id, customer_name, order_year, SUM(total_profit) AS profit
FROM IDENTIFIER(:catalog || '.' || :schema || '.agg_profit')
GROUP BY customer_id, customer_name, order_year
ORDER BY customer_id, order_year